In [ ]:
from torchrl.data import Ten

In [2]:
from pettingzoo.classic import tictactoe_v3

In [3]:
env = tictactoe_v3.env(render_mode="rgb_array")
env.reset(seed=42)

In [4]:
observation, reward, termination, truncation, info = env.last()
type(observation["observation"])

numpy.ndarray

In [5]:
observation_space = env.observation_space(env.agent_selection)
observation_space

Dict('action_mask': Box(0, 1, (9,), int8), 'observation': Box(0, 1, (3, 3, 2), int8))

In [6]:
import torch
from tensordict import TensorDict
from torchrl.data import TensorStorage

# Change later to use a config file or something
state_shape = (3, 3)  # Example for tic-tac-toe
action_shape = 9,  # 9 possible actions in tic-tac-toe
batch_size = 1  # For simplicity, we can start with a batch size of 1
max_nodes = 100  # Maximum number of nodes in the MCTS tree


device = "cuda" if torch.cuda.is_available() else "cpu"




In [25]:
tree = TensorStorage(
    storage = TensorDict(
        {
            "state": torch.zeros((max_nodes, *state_shape), device=device),
            "action": torch.zeros(max_nodes, device=device, dtype=torch.uint8),
            "value": torch.zeros(max_nodes, device=device, dtype=torch.float32),
            "visit_count": torch.zeros(max_nodes, device=device, dtype=torch.uint16),
            "parent_index": torch.full((max_nodes,), -1, dtype=torch.int64, device=device),
            "children_indices": torch.full((max_nodes, 9), -1, dtype=torch.long, device=device),  # Assuming max 9 children for tic-tac-toe
        },
        batch_size=max_nodes,
    ),
    max_size=max_nodes,
)

In [26]:
tree["value"][0]

tensor(0.)

In [27]:
root_tensordict = TensorDict(
    {
        "state": torch.zeros(state_shape, device=device),  # Placeholder for the initial state
        "action": torch.tensor(-1, device=device),  # No action taken at the root
        "value": torch.tensor(10, device=device),  # Initial value of
        "visit_count": torch.tensor(1, device=device),  # Root node has been visited once
        "parent_index": torch.tensor(-1, device=device),  # Root node has no parent
        "children_indices": torch.full((9,), -1, dtype=torch.long, device=device),  # No children for root
    }
)

tree[0] = root_tensordict

In [28]:
tree["value"][0]

tensor(10.)

In [31]:
nodes = 1  # Keep track of the number of nodes in the tree

In [ ]:
class MCTS:
    def __init__(self, max_nodes, network: torch.nn.Module = None, device="cpu"):
        self.tree: TensorStorage = TensorStorage(
            storage=TensorDict(
                {
                    "state": torch.zeros((max_nodes, *state_shape), device=device, dtype=torch.bool),
                    "action": torch.zeros(max_nodes, device=device, dtype=torch.uint8),
                    "value": torch.zeros(max_nodes, device=device, dtype=torch.float32),
                    "visit_count": torch.zeros(max_nodes, device=device, dtype=torch.int16),
                    "parent_index": torch.full((max_nodes,), -1, dtype=torch.int64, device=device),
                    "children_indices": torch.full((max_nodes, 9), -1, dtype=torch.long, device=device),  # Assuming max 9 children for tic-tac-toe
                },
                batch_size=max_nodes,
            ),
            max_size=max_nodes,
        )
        self.nodes = 1
        self.  # Keep track of the number of nodes in the tree
        
    def caluculate_ucb(self, node_index, total_visits, c_puct=1.0):
        if self.tree["visit_count"][node_index] == 0:
            return float('inf')  # Prioritize unvisited nodes
        else:
            exploitation = self.tree["value"][node_index] / self.tree["visit_count"][node_index]
            exploration = c_puct * torch.sqrt(torch.log(total_visits) / self.tree["visit_count"][node_index])
            return exploitation + exploration

    def add_node(self, parent_index, state, action, value):
        if self.nodes >= self.tree.max_size:
            raise Exception("Maximum number of nodes reached")
        
        new_index = self.nodes
        self.tree["state"][new_index] = state
        self.tree["action"][new_index] = action
        self.tree["value"][new_index] = value
        self.tree["visit_count"][new_index] = 0  # Initialize visit count to 0
        self.tree["parent_index"][new_index] = parent_index
        
        if parent_index != -1:
            # Update the parent's children indices
            for i in range(9):  # Assuming max 9 children for tic-tac-toe
                if self.tree["children_indices"][parent_index][i] == -1:
                    self.tree["children_indices"][parent_index][i] = new_index
                    break
        
        self.nodes += 1

    def backpropagate(self, node_index, value):
        # Backpropagate the value from the given node index to the root
        while node_index != -1:
            self.tree["value"][node_index] += value
            self.tree["visit_count"][node_index] += 1
            node_index = self.tree["parent_index"][node_index]

In [136]:
mcts = MCTS(max_nodes=100)

In [137]:
print(mcts.nodes)
print(mcts.tree["children_indices"][0])

1
tensor([-1, -1, -1, -1, -1, -1, -1, -1, -1])


In [114]:
mcts.add_node(
    parent_index=0,  # Assuming the root node is at index 0
    state=torch.zeros(state_shape, device=device),  # Placeholder for the new state
    action=torch.tensor(0, device=device),  # Example action
    value=torch.tensor(0.0, device=device)  # Example value
)

In [115]:
print(mcts.nodes)
print(mcts.tree["children_indices"][0])
print(mcts.tree["parent_index"][1])

2
tensor([ 1, -1, -1, -1, -1, -1, -1, -1, -1])
tensor(0)


In [116]:
mcts.backpropagate(node_index=1, value=3.0)  

In [117]:
print(mcts.tree["value"][1])
print(mcts.tree["visit_count"][1])

print(mcts.tree["value"][0])
print(mcts.tree["visit_count"][0])

tensor(3.)
tensor(1, dtype=torch.int16)
tensor(3.)
tensor(1, dtype=torch.int16)


In [118]:
def model_predict(state):
    # Placeholder for model prediction logic
    # In a real implementation, this would use a trained model to predict the value and policy
    value = torch.rand(1)  # Random value prediction
    policy = torch.rand(9)  # Random policy prediction for 9 actions
    return value, policy


In [ ]:
def add_node(tree, parent_index, state, action, value):
    